# Phase 4 — Switch Mobility Agent

This notebook runs the mobility switch agent.

Responsibilities:
- Subscribe to agent state topic
- Every switch interval, select people to switch bars
- Publish control switch commands and switch events

Rule implemented:
- Every 50 seconds, switch 25% of active people
- Choose destination bars to keep populations almost equal

In [1]:
from collections import Counter
from datetime import datetime, timezone
import json
from pathlib import Path
import random
import sys
import time
import uuid

cwd = Path.cwd()
candidate_paths = [cwd / "src", cwd.parent / "src"]
for path in candidate_paths:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher
import yaml

In [2]:
app_cfg = load_config()

config_path = Path("config.yaml")
if not config_path.exists():
    config_path = Path("../config.yaml")

raw_cfg = {}
if config_path.exists():
    raw_cfg = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}

sim_cfg = raw_cfg.get("dating_simulation", {})

RANDOM_SEED = int(sim_cfg.get("random_seed", 42))
BAR_IDS = sim_cfg.get("bar_ids", ["bar_1", "bar_2", "bar_3", "bar_4"])
SWITCH_INTERVAL_SECONDS = int(sim_cfg.get("switch_interval_seconds", 50))
SWITCH_FRACTION = float(sim_cfg.get("switch_fraction", 0.25))
BAR_BALANCE_TOLERANCE = int(sim_cfg.get("bar_balance_tolerance", 5))

random.seed(RANDOM_SEED)

TOPIC_BASE = app_cfg.mqtt.base_topic
TOPIC_STATE = f"{TOPIC_BASE}/agents/state"
TOPIC_CONTROL_SWITCH = f"{TOPIC_BASE}/control/switch"
TOPIC_EVENT_SWITCH = f"{TOPIC_BASE}/events/switch"

print({
    "topic_state": TOPIC_STATE,
    "topic_control_switch": TOPIC_CONTROL_SWITCH,
    "topic_event_switch": TOPIC_EVENT_SWITCH,
    "interval": SWITCH_INTERVAL_SECONDS,
    "fraction": SWITCH_FRACTION,
    "bars": BAR_IDS,
})

{'topic_state': 'simulated-city/agents/state', 'topic_control_switch': 'simulated-city/control/switch', 'topic_event_switch': 'simulated-city/events/switch', 'interval': 50, 'fraction': 0.25, 'bars': ['bar_1', 'bar_2', 'bar_3', 'bar_4']}


In [3]:
connector = MqttConnector(app_cfg.mqtt, client_id_suffix="agent-switch-mobility")
publisher = None
latest_people_state = {}

def on_message(client, userdata, message):
    if message.topic != TOPIC_STATE:
        return

    try:
        payload = json.loads(message.payload.decode("utf-8"))
    except Exception:
        return

    person_id = payload.get("person_id")
    bar_id = payload.get("bar_id")
    state = payload.get("state")
    timestamp = payload.get("timestamp")

    if person_id is None or bar_id is None or state is None:
        return

    latest_people_state[person_id] = {
        "bar_id": bar_id,
        "state": state,
        "timestamp": timestamp,
    }

connector.client.on_message = on_message

try:
    connector.connect()
    if connector.wait_for_connection(timeout=3.0):
        publisher = MqttPublisher(connector)
        connector.client.subscribe(TOPIC_STATE)
        print(f"Subscribed to {TOPIC_STATE}")
    else:
        print("MQTT not connected in time.")
except Exception as exc:
    print(f"MQTT unavailable ({exc})")

Subscribed to simulated-city/agents/state


In [4]:
def choose_destination(current_bar: str, bar_counts: Counter) -> str:
    candidates = [bar for bar in BAR_IDS if bar != current_bar]
    if not candidates:
        return current_bar

    min_count = min(bar_counts.get(bar, 0) for bar in candidates)
    least_loaded = [bar for bar in candidates if bar_counts.get(bar, 0) == min_count]
    return random.choice(least_loaded)


def publish_switch(person_id: int, from_bar: str, to_bar: str, reason: str) -> None:
    if publisher is None:
        return

    timestamp = datetime.now(timezone.utc).isoformat()

    control_payload = {
        "event_id": str(uuid.uuid4()),
        "person_id": person_id,
        "from_bar_id": from_bar,
        "to_bar_id": to_bar,
        "reason": reason,
        "timestamp": timestamp,
    }

    event_payload = dict(control_payload)

    publisher.publish_json(TOPIC_CONTROL_SWITCH, json.dumps(control_payload), qos=0, retain=False)
    publisher.publish_json(TOPIC_EVENT_SWITCH, json.dumps(event_payload), qos=0, retain=False)


def run_switch_cycle() -> None:
    if not latest_people_state:
        print("No state data yet. Waiting for agent state messages...")
        return

    active_people = [
        (person_id, data)
        for person_id, data in latest_people_state.items()
        if data.get("state") != "removed"
    ]

    if not active_people:
        print("No active people available for switching.")
        return

    switch_count = max(1, int(len(active_people) * SWITCH_FRACTION))
    selected = random.sample(active_people, k=min(switch_count, len(active_people)))

    bar_counts = Counter(data["bar_id"] for _, data in latest_people_state.items())

    switched = 0
    for person_id, data in selected:
        from_bar = data["bar_id"]
        to_bar = choose_destination(from_bar, bar_counts)

        if to_bar == from_bar:
            continue

        publish_switch(person_id, from_bar, to_bar, "periodic_balance_switch")

        latest_people_state[person_id]["bar_id"] = to_bar
        bar_counts[from_bar] -= 1
        bar_counts[to_bar] += 1
        switched += 1

    max_count = max(bar_counts.values()) if bar_counts else 0
    min_count = min(bar_counts.values()) if bar_counts else 0
    delta = max_count - min_count

    print(f"Switch cycle complete: switched={switched}, balance_delta={delta}, tolerance={BAR_BALANCE_TOLERANCE}")


# Run one cycle manually to verify behavior
run_switch_cycle()

No state data yet. Waiting for agent state messages...


In [5]:
def run_scheduler(cycles: int = 3):
    for idx in range(cycles):
        print(f"\nCycle {idx + 1}/{cycles}...")
        run_switch_cycle()
        if idx < cycles - 1:
            time.sleep(SWITCH_INTERVAL_SECONDS)


# Optional: uncomment to run a timed scheduler
# run_scheduler(cycles=3)

In [6]:
if connector is not None:
    connector.disconnect()
    print("Disconnected MQTT client.")

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


Disconnected MQTT client.
